# Class-3 Forensic Analysis / Evidence Layer

**The Class-3 inference is already complete and frozen.** This notebook does not retrain the
model, does not refit any preprocessing, does not change the threshold, and does not modify the
existing Class-3 scores. It continues from `05_Inference_Forensic_Analysis.ipynb`, loading the
already-produced `outputs/unknown_transaction_risk_scores.csv` and
`outputs/top_unknown_forensic_cases.csv`, and builds a descriptive forensic evidence layer for the
top candidate cases.

**Class 3 = UNKNOWN.** A high illicit-risk score does not establish that a transaction is illicit
or fraudulent. Throughout this notebook: "model-based illicit-risk score", "high-risk candidate",
"supporting behavioural indicators", and "candidate for forensic review" are used. "Confirmed
fraud", "confirmed illicit transaction", and "proof of fraud" are never used.

## 0. Imports and paths

In [1]:
import time, os, json
import numpy as np
import pandas as pd
from scipy import stats

t0 = time.time()
def log(msg): print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)

DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"
PROCESSED_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed"
ARTIFACT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42"
OUTPUTS_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs"
print("Paths configured.")

Paths configured.


## Step 1 — Use the existing Class-3 scores

Loads the frozen scores exactly as produced by `05_Inference_Forensic_Analysis.ipynb`. The scores
are **not** regenerated here.

In [2]:
print("="*70)
print("STEP 1: LOAD EXISTING CLASS-3 SCORES")
print("="*70)

risk_scores = pd.read_csv(os.path.join(OUTPUTS_DIR, "unknown_transaction_risk_scores.csv"))
assert len(risk_scores) == 157205, f"Row count mismatch: {len(risk_scores)}"
assert risk_scores["txId"].duplicated().sum() == 0, "Duplicate txId found!"
assert risk_scores["illicit_risk_score"].isna().sum() == 0, "NaN scores found!"
assert (risk_scores["illicit_risk_score"] >= 0).all() and (risk_scores["illicit_risk_score"] <= 1).all(), "Scores out of [0,1]!"
print(f"[PASS] unknown_transaction_risk_scores.csv: {len(risk_scores)} rows, no dup txId, no NaN, scores in [0,1].")

top20_scores = pd.read_csv(os.path.join(OUTPUTS_DIR, "top_unknown_forensic_cases.csv"))
assert len(top20_scores) == 20
print(f"[PASS] top_unknown_forensic_cases.csv loaded: {len(top20_scores)} rows.")

STEP 1: LOAD EXISTING CLASS-3 SCORES
[PASS] unknown_transaction_risk_scores.csv: 157205 rows, no dup txId, no NaN, scores in [0,1].
[PASS] top_unknown_forensic_cases.csv loaded: 20 rows.


## Step 2 — Select forensic cases

The existing top-20 ranking by `illicit_risk_score` is preserved exactly — no new threshold or
model is used to select cases.

In [3]:
print("\n" + "="*70)
print("STEP 2: SELECT FORENSIC CASES (existing top-20 ranking preserved)")
print("="*70)

top20 = top20_scores[["rank", "txId", "Time step", "illicit_risk_score", "risk_level"]].copy()
print(top20.to_string(index=False))


STEP 2: SELECT FORENSIC CASES (existing top-20 ranking preserved)
 rank      txId  Time step  illicit_risk_score risk_level
    1  99942047         42                 1.0       HIGH
    2  99942248         42                 1.0       HIGH
    3  99942236         42                 1.0       HIGH
    4  99942037         42                 1.0       HIGH
    5  99943929         42                 1.0       HIGH
    6  99942297         42                 1.0       HIGH
    7  99942066         42                 1.0       HIGH
    8 403235564         28                 1.0       HIGH
    9 298977429         30                 1.0       HIGH
   10  99942303         42                 1.0       HIGH
   11 112618841         32                 1.0       HIGH
   12  99943087         42                 1.0       HIGH
   13  99943310         42                 1.0       HIGH
   14 244628157         15                 1.0       HIGH
   15  99942183         42                 1.0       HIGH
   16

## Step 3 — Retrieve final B features

Rebuilds the raw (pre-scaling) B-representation feature values for the top-20 transaction IDs,
using the exact same feature construction as `05_Inference_Forensic_Analysis.ipynb`. The resulting
186-column ordering is checked against the saved `feature_columns.json` and the notebook stops if
it does not match exactly.

In [4]:
print("\n" + "="*70)
print("STEP 3: RETRIEVE FINAL B FEATURES (same construction as the inference notebook)")
print("="*70)

with open(os.path.join(ARTIFACT_DIR, "feature_columns.json")) as f:
    saved_feature_cols = json.load(f)

tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv")
tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
tx_data = tx_features.merge(tx_classes, on="txId", how="left")
tx_data = tx_data.sort_values("txId", kind="mergesort").reset_index(drop=True)
assert len(tx_data) == 203769

model_feature_cols = [c for c in tx_features.columns if c != "txId"]
REMOVE_13 = [
    "Local_feature_49", "Local_feature_65", "Local_feature_30", "Local_feature_29",
    "Local_feature_23", "Local_feature_47", "Local_feature_25", "Local_feature_43",
    "Local_feature_41", "Local_feature_54", "Local_feature_53", "Local_feature_55",
    "Local_feature_18",
]
a2_cols = [c for c in model_feature_cols if c not in REMOVE_13]
assert len(a2_cols) == 170

augmented_features = [
    "in_txs_degree", "out_txs_degree", "total_BTC", "fees", "size",
    "num_input_addresses", "num_output_addresses",
    "in_BTC_min", "in_BTC_max", "in_BTC_mean", "in_BTC_median", "in_BTC_total",
    "out_BTC_min", "out_BTC_max", "out_BTC_mean", "out_BTC_median", "out_BTC_total",
]
NEW_14 = ["FIN3","FIN4","FIN5","FIN6","STR1","STR2","TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","NET1","NET2"]

new_feats = pd.read_parquet(os.path.join(PROCESSED_DIR, "transaction_time_safe_features_v1.parquet"))
merged = tx_data[["txId", "class"] + a2_cols].merge(
    new_feats[["txId"] + NEW_14 + ["TMP6_is_first_appearance"]], on="txId", how="left"
)
assert len(merged) == len(tx_data)
merged["augmented_features_missing"] = merged[augmented_features].isna().all(axis=1).astype(int)

final_feature_cols_b = a2_cols + ["augmented_features_missing"] + NEW_14 + ["TMP6_is_first_appearance"]
assert final_feature_cols_b == saved_feature_cols, "STOP: feature ordering mismatch vs feature_columns.json!"
print(f"[PASS] Feature construction matches feature_columns.json exactly ({len(final_feature_cols_b)} features).")

merged_indexed = merged.set_index("txId")
# "Time step" is one of the a2_cols (already present in top20 from the frozen scores file) --
# drop it from the merge source to avoid a Time step_x/Time step_y collision, after confirming
# the two sources agree.
_check = top20.merge(merged_indexed[["Time step"]].reset_index(), on="txId", how="left")
assert (_check["Time step_x"] == _check["Time step_y"]).all(), "Time step mismatch between frozen scores and rebuilt features!"
top20_raw = top20.merge(merged_indexed.drop(columns=["Time step"]).reset_index(), on="txId", how="left")
assert top20_raw.shape[0] == 20
print(f"[PASS] Retrieved raw B-representation feature values for all {len(top20_raw)} top-20 transactions.")


STEP 3: RETRIEVE FINAL B FEATURES (same construction as the inference notebook)


[PASS] Feature construction matches feature_columns.json exactly (186 features).


[PASS] Retrieved raw B-representation feature values for all 20 top-20 transactions.


## Steps 4–7 — Financial, Structural, Temporal, Network indicators

Actual values only — no automatic "high/low = suspicious" labelling anywhere in this notebook.

In [5]:
print("\n" + "="*70)
print("STEP 4-7: FINANCIAL / STRUCTURAL / TEMPORAL / NETWORK INDICATORS (top-20 preview)")
print("="*70)

FINANCIAL_COLS = ["total_BTC","fees","size","FIN3","FIN4","FIN5","FIN6",
                   "in_BTC_min","in_BTC_max","in_BTC_mean","in_BTC_median","in_BTC_total",
                   "out_BTC_min","out_BTC_max","out_BTC_mean","out_BTC_median","out_BTC_total"]
STRUCTURAL_COLS = ["STR1","STR2","num_input_addresses","num_output_addresses","in_txs_degree","out_txs_degree"]
TEMPORAL_COLS = ["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"]
NETWORK_COLS = ["NET1","NET2"]
ALL_INDICATOR_COLS = FINANCIAL_COLS + STRUCTURAL_COLS + TEMPORAL_COLS + NETWORK_COLS

for c in ALL_INDICATOR_COLS:
    assert c in top20_raw.columns, f"Missing indicator column: {c}"
print(f"All {len(ALL_INDICATOR_COLS)} indicator columns present (Financial={len(FINANCIAL_COLS)}, "
      f"Structural={len(STRUCTURAL_COLS)}, Temporal={len(TEMPORAL_COLS)}, Network={len(NETWORK_COLS)}).")
print("\nSample (rank 1):")
r1 = top20_raw.iloc[0]
print("  Financial:", {c: r1[c] for c in FINANCIAL_COLS[:7]})
print("  Structural:", {c: r1[c] for c in STRUCTURAL_COLS})
print("  Temporal:", {c: r1[c] for c in TEMPORAL_COLS})
print("  Network:", {c: r1[c] for c in NETWORK_COLS})


STEP 4-7: FINANCIAL / STRUCTURAL / TEMPORAL / NETWORK INDICATORS (top-20 preview)
All 32 indicator columns present (Financial=17, Structural=6, Temporal=7, Network=2).

Sample (rank 1):
  Financial: {'total_BTC': 0.04393081, 'fees': 0.00138368, 'size': 191.0, 'FIN3': 1.0, 'FIN4': 1.0, 'FIN5': 0.0314968, 'FIN6': 7.2443977e-06}
  Structural: {'STR1': 1.0, 'STR2': 0, 'num_input_addresses': 1.0, 'num_output_addresses': 1.0, 'in_txs_degree': 1.0, 'out_txs_degree': 0.0}
  Temporal: {'TMP1': 0, 'TMP2': 0, 'TMP4': 0, 'TMP5': 13, 'TMP6': <NA>, 'TMP7': 1, 'TMP6_is_first_appearance': 1}
  Network: {'NET1': 0, 'NET2': 13}


## Step 8 — Reference context

Compares each candidate's indicator values against the **labelled reference population** (known
Class 1/2 transactions only — Class 3 is never used as a reference). Percentile is computed within
all labelled transactions; medians are also reported separately for labelled illicit and labelled
licit subsets. These are **associations observed in labelled data**, not proof of illicit activity
for any Class-3 transaction.

In [6]:
print("\n" + "="*70)
print("STEP 8: REFERENCE CONTEXT (labelled reference population)")
print("="*70)

labelled = merged[merged["class"].isin([1, 2])].copy()
illicit_ref = labelled[labelled["class"] == 1]
licit_ref = labelled[labelled["class"] == 2]
print(f"Labelled reference population: {len(labelled)} (illicit={len(illicit_ref)}, licit={len(licit_ref)})")
print("Class-3 transactions are NEVER used as the reference population.")

def percentile_of(value, reference_series):
    ref = reference_series.dropna().values
    if pd.isna(value) or len(ref) == 0:
        return None
    return float(stats.percentileofscore(ref, value, kind="mean"))

reference_stats = {}
for c in ALL_INDICATOR_COLS:
    reference_stats[c] = {
        "all_median": labelled[c].median(),
        "illicit_median": illicit_ref[c].median(),
        "licit_median": licit_ref[c].median(),
    }

percentile_rows = []
for _, row in top20_raw.iterrows():
    prow = {"rank": row["rank"], "txId": row["txId"]}
    for c in ["FIN3","FIN4","FIN5","NET1","NET2","TMP4","TMP5"]:
        prow[f"{c}_value"] = row[c]
        prow[f"{c}_percentile_all_labelled"] = percentile_of(row[c], labelled[c])
        prow[f"{c}_median_all_labelled"] = reference_stats[c]["all_median"]
        prow[f"{c}_median_illicit_labelled"] = reference_stats[c]["illicit_median"]
        prow[f"{c}_median_licit_labelled"] = reference_stats[c]["licit_median"]
    percentile_rows.append(prow)
percentile_df = pd.DataFrame(percentile_rows)
print(f"\nComputed percentile/median reference context for {len(percentile_df)} cases across 7 headline indicators.")
print("Example (rank 1, FIN3 and NET2):")
r1p = percentile_df.iloc[0]
print(f"  FIN3 value={r1p['FIN3_value']:.4g}  percentile(all labelled)={r1p['FIN3_percentile_all_labelled']:.1f}  "
      f"median(all)={r1p['FIN3_median_all_labelled']:.4g}  median(illicit)={r1p['FIN3_median_illicit_labelled']:.4g}  "
      f"median(licit)={r1p['FIN3_median_licit_labelled']:.4g}")
print(f"  NET2 value={r1p['NET2_value']:.4g}  percentile(all labelled)={r1p['NET2_percentile_all_labelled']:.1f}  "
      f"median(all)={r1p['NET2_median_all_labelled']:.4g}  median(illicit)={r1p['NET2_median_illicit_labelled']:.4g}  "
      f"median(licit)={r1p['NET2_median_licit_labelled']:.4g}")
print("\nNOTE: these are associations observed in labelled data. They do not establish illicit activity for Class-3 transactions.")


STEP 8: REFERENCE CONTEXT (labelled reference population)
Labelled reference population: 46564 (illicit=4545, licit=42019)
Class-3 transactions are NEVER used as the reference population.



Computed percentile/median reference context for 20 cases across 7 headline indicators.
Example (rank 1, FIN3 and NET2):
  FIN3 value=1  percentile(all labelled)=73.3  median(all)=1  median(illicit)=1  median(licit)=0.9998
  NET2 value=13  percentile(all labelled)=65.4  median(all)=1  median(illicit)=5  median(licit)=1

NOTE: these are associations observed in labelled data. They do not establish illicit activity for Class-3 transactions.


## Step 9 — Descriptive forensic indicators

A structured profile per transaction. These are explicitly labelled **supporting behavioural
indicators**, not model explanations — no formal attribution method (SHAP, GNNExplainer, gradient
attribution) has been applied in this project yet (see Step 10).

In [7]:
print("\n" + "="*70)
print("STEP 9: DESCRIPTIVE FORENSIC PROFILES (SUPPORTING BEHAVIOURAL INDICATORS)")
print("="*70)

def fmt(v):
    if pd.isna(v):
        return "N/A (no qualifying prior activity)"
    if isinstance(v, (int, np.integer)):
        return str(v)
    return f"{v:.6g}"

def build_profile(row):
    lines = []
    lines.append("-"*36)
    lines.append("Transaction")
    lines.append("-"*36)
    lines.append(f"Transaction ID: {row['txId']}")
    lines.append(f"Time step: {row['Time step']}")
    lines.append(f"Illicit-risk score: {row['illicit_risk_score']:.4f}")
    lines.append(f"Risk level: {row['risk_level']}")
    lines.append("")
    lines.append("-"*36)
    lines.append("Financial indicators")
    lines.append("-"*36)
    lines.append(f"- Total BTC: {fmt(row['total_BTC'])}")
    lines.append(f"- Fees: {fmt(row['fees'])}")
    lines.append(f"- Transaction size: {fmt(row['size'])}")
    lines.append(f"- Largest input share: {fmt(row['FIN3'])}")
    lines.append(f"- Largest output share: {fmt(row['FIN4'])}")
    lines.append(f"- Fee-to-value ratio: {fmt(row['FIN5'])}")
    lines.append(f"- Fee rate: {fmt(row['FIN6'])}")
    lines.append("")
    lines.append("-"*36)
    lines.append("Structural indicators")
    lines.append("-"*36)
    lines.append(f"- Input addresses: {fmt(row['num_input_addresses'])}")
    lines.append(f"- Output addresses: {fmt(row['num_output_addresses'])}")
    lines.append(f"- Input/output ratio: {fmt(row['STR1'])}")
    lines.append(f"- Same-transaction address reuse: {bool(row['STR2']) if pd.notna(row['STR2']) else 'N/A'}")
    lines.append(f"- Incoming transaction degree: {fmt(row['in_txs_degree'])}")
    lines.append(f"- Outgoing transaction degree: {fmt(row['out_txs_degree'])}")
    lines.append("")
    lines.append("-"*36)
    lines.append("Temporal indicators")
    lines.append("-"*36)
    lines.append(f"- Input-address age: {fmt(row['TMP1'])}")
    lines.append(f"- Output-address age: {fmt(row['TMP2'])}")
    lines.append(f"- Prior sender transactions: {fmt(row['TMP4'])}")
    lines.append(f"- Prior receiver transactions: {fmt(row['TMP5'])}")
    lines.append(f"- Time since last activity: {fmt(row['TMP6'])}")
    lines.append(f"- Distinct active time steps: {fmt(row['TMP7'])}")
    lines.append(f"- First appearance indicator: {fmt(row['TMP6_is_first_appearance'])}")
    lines.append("")
    lines.append("-"*36)
    lines.append("Network indicators")
    lines.append("-"*36)
    lines.append(f"- Historical input-linked transaction count: {fmt(row['NET1'])}")
    lines.append(f"- Historical output-linked transaction count: {fmt(row['NET2'])}")
    lines.append("")
    lines.append("These are SUPPORTING BEHAVIOURAL INDICATORS, not model explanations. No formal")
    lines.append("attribution method (e.g. SHAP, GNNExplainer, gradient attribution) has been applied.")
    lines.append("They support forensic review but do not establish illicit activity.")
    return "\n".join(lines)

all_profiles = [build_profile(row) for _, row in top20_raw.iterrows()]
print(f"Generated {len(all_profiles)} structured forensic profiles.")
print("\n\n".join(all_profiles[:2]))
print(f"\n... ({len(all_profiles)-2} more profiles generated, not printed here for brevity) ...")


STEP 9: DESCRIPTIVE FORENSIC PROFILES (SUPPORTING BEHAVIOURAL INDICATORS)
Generated 20 structured forensic profiles.
------------------------------------
Transaction
------------------------------------
Transaction ID: 99942047
Time step: 42
Illicit-risk score: 1.0000
Risk level: HIGH

------------------------------------
Financial indicators
------------------------------------
- Total BTC: 0.0439308
- Fees: 0.00138368
- Transaction size: 191
- Largest input share: 1
- Largest output share: 1
- Fee-to-value ratio: 0.0314968
- Fee rate: 7.2444e-06

------------------------------------
Structural indicators
------------------------------------
- Input addresses: 1
- Output addresses: 1
- Input/output ratio: 1
- Same-transaction address reuse: False
- Incoming transaction degree: 1
- Outgoing transaction degree: 0

------------------------------------
Temporal indicators
------------------------------------
- Input-address age: 0
- Output-address age: 0
- Prior sender transactions: 0
- 

## Step 10 — Not yet model explanations

The indicators above are deliberately labelled **supporting behavioural indicators**, not model
explanations. A feature being unusual for a given transaction does not establish that the model
used that feature to produce its score. Formal attribution (SHAP, GNNExplainer, gradient
attribution) is explicitly out of scope for this notebook and has **not** been implemented.

## Step 11 — Investigate the top-20 pattern

The prior inference notebook noted that several of the top-20 highest-scoring cases appeared to
share a similar fingerprint. This section checks that observation directly and quantitatively
rather than assuming it.

In [8]:
print("\n" + "="*70)
print("STEP 11: INVESTIGATE THE TOP-20 RECURRING PATTERN")
print("="*70)

n_single_input = int((top20_raw["num_input_addresses"] == 1).sum())
n_single_output = int((top20_raw["num_output_addresses"] == 1).sum())
n_first_appearance = int((top20_raw["TMP6_is_first_appearance"] == 1).sum())
net2_dist = top20_raw["NET2"].describe()
timestep_dist = top20_raw["Time step"].value_counts().sort_index()

print(f"1. Transactions with exactly 1 input address:  {n_single_input} / 20")
print(f"2. Transactions with exactly 1 output address: {n_single_output} / 20")
print(f"3. First-appearance transactions:              {n_first_appearance} / 20")
print(f"4. NET2 distribution among top 20:\n{net2_dist.to_string()}")
print(f"5. Time step distribution among top 20:\n{timestep_dist.to_string()}")

pct_single_input_ref = float((labelled["num_input_addresses"] == 1).mean() * 100)
pct_single_output_ref = float((labelled["num_output_addresses"] == 1).mean() * 100)
pct_first_appearance_ref = float((labelled["TMP6_is_first_appearance"] == 1).mean() * 100)
pct_single_input_illicit = float((illicit_ref["num_input_addresses"] == 1).mean() * 100)
pct_single_output_illicit = float((illicit_ref["num_output_addresses"] == 1).mean() * 100)
pct_first_appearance_illicit = float((illicit_ref["TMP6_is_first_appearance"] == 1).mean() * 100)

print(f"\n6. Commonality in labelled reference population (n={len(labelled)}):")
print(f"   Single input address:  {pct_single_input_ref:.2f}% of all labelled  |  {pct_single_input_illicit:.2f}% of labelled illicit")
print(f"   Single output address: {pct_single_output_ref:.2f}% of all labelled  |  {pct_single_output_illicit:.2f}% of labelled illicit")
print(f"   First appearance:      {pct_first_appearance_ref:.2f}% of all labelled  |  {pct_first_appearance_illicit:.2f}% of labelled illicit")

combo_mask_ref = (labelled["num_input_addresses"] == 1) & (labelled["num_output_addresses"] == 1) & (labelled["TMP6_is_first_appearance"] == 1)
pct_combo_ref = float(combo_mask_ref.mean() * 100)
combo_mask_top20 = (top20_raw["num_input_addresses"] == 1) & (top20_raw["num_output_addresses"] == 1) & (top20_raw["TMP6_is_first_appearance"] == 1)
n_combo_top20 = int(combo_mask_top20.sum())
print(f"\nFull combination (1-in AND 1-out AND first-appearance): {n_combo_top20}/20 in top-20 vs {pct_combo_ref:.2f}% in labelled reference population.")
print("\nReported as: \"Observed recurring behavioural fingerprint among the highest-scoring candidates.\"")
print("This is a descriptive finding, NOT a claim that the model detects this pattern as fraud.")


STEP 11: INVESTIGATE THE TOP-20 RECURRING PATTERN
1. Transactions with exactly 1 input address:  20 / 20
2. Transactions with exactly 1 output address: 20 / 20
3. First-appearance transactions:              20 / 20
4. NET2 distribution among top 20:
count         20.0
mean          25.3
std      69.569428
min            0.0
25%           6.25
50%           13.0
75%           13.0
max          320.0
5. Time step distribution among top 20:
Time step
8      1
9      1
15     1
28     1
30     1
32     1
42    12
46     2

6. Commonality in labelled reference population (n=46564):
   Single input address:  52.83% of all labelled  |  89.11% of labelled illicit
   Single output address: 12.28% of all labelled  |  64.91% of labelled illicit
   First appearance:      61.71% of all labelled  |  89.57% of labelled illicit

Full combination (1-in AND 1-out AND first-appearance): 20/20 in top-20 vs 7.93% in labelled reference population.

Reported as: "Observed recurring behavioural fingerprint a

## Step 12 — Output table: `top20_forensic_case_profiles.csv`

Raw indicator values for all 20 cases, grouped by category as specified.

In [9]:
print("\n" + "="*70)
print("STEP 12: SAVE outputs/top20_forensic_case_profiles.csv")
print("="*70)

profile_cols = (["rank","txId","Time step","illicit_risk_score","risk_level"] +
                 ["total_BTC","fees","size","FIN3","FIN4","FIN5","FIN6"] +
                 ["num_input_addresses","num_output_addresses","STR1","STR2","in_txs_degree","out_txs_degree"] +
                 ["TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","TMP6_is_first_appearance"] +
                 ["NET1","NET2"])
profiles_df = top20_raw[profile_cols].copy()
profiles_path = os.path.join(OUTPUTS_DIR, "top20_forensic_case_profiles.csv")
profiles_df.to_csv(profiles_path, index=False)
assert len(profiles_df) == 20
print(f"Saved: {profiles_path}  ({len(profiles_df)} rows, {len(profile_cols)} columns)")


STEP 12: SAVE outputs/top20_forensic_case_profiles.csv
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_forensic_case_profiles.csv  (20 rows, 27 columns)


## Step 13 — Human-readable forensic report: `top20_forensic_case_report.csv`

Concise, non-sensational natural-language summaries per case, plus the reference-context
comparison and a full narrative sentence in the style specified.

In [10]:
print("\n" + "="*70)
print("STEP 13: SAVE outputs/top20_forensic_case_report.csv")
print("="*70)

def fmt2(v, nd=4):
    return "N/A" if pd.isna(v) else f"{v:.{nd}g}"

report_rows = []
for _, row in top20_raw.iterrows():
    financial_summary = (f"Total BTC = {fmt2(row['total_BTC'])}, fees = {fmt2(row['fees'])}, "
                          f"size = {fmt2(row['size'])} bytes; largest input share = {fmt2(row['FIN3'])}, "
                          f"largest output share = {fmt2(row['FIN4'])}, fee-to-value ratio = {fmt2(row['FIN5'])}, "
                          f"fee rate = {fmt2(row['FIN6'])}.")
    structural_summary = (f"{fmt2(row['num_input_addresses'],3)} input address(es), "
                           f"{fmt2(row['num_output_addresses'],3)} output address(es) "
                           f"(input/output ratio = {fmt2(row['STR1'])}); "
                           f"same-transaction address reuse = {bool(row['STR2']) if pd.notna(row['STR2']) else 'N/A'}; "
                           f"incoming tx degree = {fmt2(row['in_txs_degree'],3)}, outgoing tx degree = {fmt2(row['out_txs_degree'],3)}.")
    temporal_summary = (f"{'First-ever appearance' if row['TMP6_is_first_appearance']==1 else 'Not a first appearance'} "
                         f"at time step {row['Time step']}; input-address age = {fmt2(row['TMP1'])}, "
                         f"output-address age = {fmt2(row['TMP2'])}; {fmt2(row['TMP4'])} prior sender transaction(s), "
                         f"{fmt2(row['TMP5'])} prior receiver transaction(s); active across {fmt2(row['TMP7'])} distinct time step(s); "
                         f"time since last activity = {fmt2(row['TMP6'])}.")
    network_summary = (f"{fmt2(row['NET1'])} historical input-linked transaction(s), "
                        f"{fmt2(row['NET2'])} historical output-linked transaction(s).")
    fin3_pct = percentile_of(row["FIN3"], labelled["FIN3"])
    net2_pct = percentile_of(row["NET2"], labelled["NET2"])
    reference_context = (
        f"Single-input/single-output first-appearance transactions comprise {pct_combo_ref:.2f}% of the labelled "
        f"reference population. This transaction's largest-input share (FIN3={fmt2(row['FIN3'])}) is at the "
        f"{fin3_pct:.1f}th percentile and its output-linked count (NET2={fmt2(row['NET2'])}) is at the "
        f"{net2_pct:.1f}th percentile of all labelled transactions. These are associations observed in labelled "
        f"data, not proof of illicit activity for this Class-3 transaction."
    )
    narrative = (
        f"Transaction {row['txId']} received an illicit-risk score of {row['illicit_risk_score']:.4f} and was "
        f"therefore flagged {row['risk_level']} under the previously validated 0.90 screening threshold. "
        f"The transaction has {fmt2(row['num_input_addresses'],3)} input address(es) and "
        f"{fmt2(row['num_output_addresses'],3)} output address(es), with a largest-input share of {fmt2(row['FIN3'])}. "
        f"Its input-address age is {fmt2(row['TMP1'])} and it has {fmt2(row['TMP4'])} prior sender transaction(s). "
        f"These characteristics are presented as supporting behavioural indicators for forensic review; they do "
        f"not establish illicit activity."
    )
    report_rows.append({
        "Transaction ID": row["txId"],
        "Risk Score": round(float(row["illicit_risk_score"]), 4),
        "Risk Level": row["risk_level"],
        "Financial Summary": financial_summary,
        "Structural Summary": structural_summary,
        "Temporal Summary": temporal_summary,
        "Network Summary": network_summary,
        "Reference Context": reference_context,
        "Narrative": narrative,
    })
report_df = pd.DataFrame(report_rows)
report_path = os.path.join(OUTPUTS_DIR, "top20_forensic_case_report.csv")
report_df.to_csv(report_path, index=False)
assert len(report_df) == 20
print(f"Saved: {report_path}  ({len(report_df)} rows)")
print("\nExample narrative (rank 1):")
print(report_df.iloc[0]["Narrative"])


STEP 13: SAVE outputs/top20_forensic_case_report.csv
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_forensic_case_report.csv  (20 rows)

Example narrative (rank 1):
Transaction 99942047 received an illicit-risk score of 1.0000 and was therefore flagged HIGH under the previously validated 0.90 screening threshold. The transaction has 1 input address(es) and 1 output address(es), with a largest-input share of 1. Its input-address age is 0 and it has 0 prior sender transaction(s). These characteristics are presented as supporting behavioural indicators for forensic review; they do not establish illicit activity.


## Step 14 — Important limitations

1. **Class 3 labels are unknown.** No Class-3 transaction has a verified ground-truth label
   anywhere in this project.
2. **High-risk scores are model outputs, not confirmed fraud labels.** `illicit_risk_score` and
   `risk_level = HIGH` reflect the trained model's softmax output at the validated threshold — they
   are not a determination of illicit activity.
3. **Behavioural indicators are descriptive, not attributive, unless formal model attribution is
   performed.** Nothing in this notebook establishes that any specific feature caused the model's
   score. No SHAP, GNNExplainer, or gradient-attribution method has been applied yet.
4. **Reference comparisons show association, not causation.** Percentiles and medians computed
   against the labelled population describe how common a value is among known licit/illicit
   transactions — they do not imply that value caused a label or a score.
5. **The final model is transductive on the full transaction graph.** Class-3 (unknown) nodes
   participate in message passing during inference because they are real transactions in the graph,
   even though they never contributed to training loss or evaluation.
6. **The 0.90 threshold was fixed from the validated labelled-data protocol** (coarse+fine grid
   search on validation predictions only) and was **not** tuned or adjusted using any Class-3
   result.

## Final verification

Confirms the top-20 identifiers match the frozen scores file, checks the new output table for
NaN/Inf, and prints the completion summary.

In [11]:
print("\n" + "="*70)
print("FINAL VERIFICATION")
print("="*70)

assert list(profiles_df["txId"].values) == list(top20_scores["txId"].values), "Top-20 IDs do not match existing top-20 score file!"
print("[PASS] Top-20 IDs match existing top_unknown_forensic_cases.csv exactly (same order).")

for c in ["illicit_risk_score"]:
    assert not profiles_df[c].isna().any() and not np.isinf(profiles_df[c].values).any()
print("[PASS] No NaN/Inf in illicit_risk_score column of the new profile table.")

print(f"\nCLASS-3 FORENSIC ANALYSIS COMPLETE")
print(f"Class-3 transactions analyzed (loaded from frozen scores): {len(risk_scores)}")
print(f"Top-20 cases analyzed: {len(top20_raw)}")
print(f"Output files created:")
print(f"  - {profiles_path}")
print(f"  - {report_path}")
print(f"No training or retraining occurred in this notebook.")
print(f"\nTotal runtime: {time.time()-t0:.1f}s")


FINAL VERIFICATION
[PASS] Top-20 IDs match existing top_unknown_forensic_cases.csv exactly (same order).
[PASS] No NaN/Inf in illicit_risk_score column of the new profile table.

CLASS-3 FORENSIC ANALYSIS COMPLETE
Class-3 transactions analyzed (loaded from frozen scores): 157205
Top-20 cases analyzed: 20
Output files created:
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_forensic_case_profiles.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top20_forensic_case_report.csv
No training or retraining occurred in this notebook.

Total runtime: 6.3s
